# **FEDFormer**

это ещё один «специальный трансформер» для временных рядов, но с упором на частоты (Fourier) и разделение ряда на тренд + сезонность.
Полное название: Frequency Enhanced Decomposed Transformer

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    Add, Lambda, AveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("Brent.csv")
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2015-10-26 10:00:00,48.05,48.12,47.89,48.09,48.28815,48.15758,48.06107,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
1,2015-10-26 11:00:00,48.10,48.36,48.00,48.30,48.26098,48.12601,48.05886,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
2,2015-10-26 12:00:00,48.30,48.35,48.18,48.30,48.23552,48.11588,48.05209,0,0,...,0,-1,three_color,0,0,NaN,NaN,NaN,0,0
3,2015-10-26 13:00:00,48.30,48.34,48.05,48.09,48.21971,48.10765,48.04267,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
4,2015-10-26 14:00:00,48.11,48.28,47.97,48.07,48.19550,48.09732,48.07014,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36239,2025-10-21 23:00:00,61.55,61.73,61.55,61.65,61.17612,61.14427,61.29423,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36240,2025-10-22 09:00:00,62.27,62.66,62.26,62.46,61.16565,61.21749,61.31439,1,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36241,2025-10-22 10:00:00,62.45,62.49,62.19,62.35,61.13752,61.23530,61.35151,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36242,2025-10-22 11:00:00,62.35,62.55,62.25,62.36,61.14002,61.25526,61.40921,0,0,...,0,0,NaN,1,0,61.52,1.0,0.3,0,0


In [ ]:
scale_cols = [
    "Open", "High", "Low", "Close",
    "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
    "AO",
    "AddOn_Anchor_Level", "AddOn_Size_Pct"
]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])


In [ ]:
columns = [
    "AddOn_Anchor_Level",
    "AddOn_Anchor_IsUp",
    "AddOn_Size_Pct",

]

df = df.dropna(subset=columns).reset_index(drop=True)

In [ ]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2015-10-26 19:00:00,-0.851161,-0.846918,-0.847880,-0.847430,-0.838401,-0.838317,-0.842386,1,0,...,0,0,NaN,0,1,-0.826423,0.0,0.0,1,1
1,2015-10-26 20:00:00,-0.847015,-0.850641,-0.854530,-0.856136,-0.838041,-0.839319,-0.845333,0,0,...,0,0,NaN,0,0,-0.826423,0.0,0.0,0,0
2,2015-10-26 21:00:00,-0.855721,-0.854365,-0.855362,-0.859453,-0.837932,-0.841183,-0.846775,0,0,...,0,0,NaN,1,0,-0.826423,0.0,0.0,0,0
3,2015-10-26 22:00:00,-0.859038,-0.863881,-0.859102,-0.863599,-0.838056,-0.843231,-0.847305,0,0,...,1,-1,saucer,0,0,-0.826423,0.0,0.0,0,0
4,2015-10-26 23:00:00,-0.863184,-0.867191,-0.864090,-0.870232,-0.838778,-0.844450,-0.848769,0,0,...,0,0,NaN,0,0,-0.826423,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36230,2025-10-21 23:00:00,-0.279436,-0.278858,-0.272236,-0.275705,-0.296228,-0.296875,-0.290313,0,0,...,0,0,NaN,0,0,-0.264634,1.0,0.0,0,0
36231,2025-10-22 09:00:00,-0.249585,-0.240381,-0.242727,-0.242123,-0.296664,-0.293829,-0.289475,1,0,...,0,0,NaN,0,0,-0.264634,1.0,0.0,0,0
36232,2025-10-22 10:00:00,-0.242123,-0.247414,-0.245636,-0.246683,-0.297835,-0.293088,-0.287931,0,0,...,0,0,NaN,0,0,-0.264634,1.0,0.0,0,0
36233,2025-10-22 11:00:00,-0.246269,-0.244932,-0.243142,-0.246269,-0.297731,-0.292258,-0.285532,0,0,...,0,0,NaN,1,0,-0.264634,1.0,0.0,0,0


In [ ]:
# 1) Читаем данные
df = pd.read_csv("Brent.csv")

# 2) сортировка по времени, чтобы shift(-H) был в будущее
df = df.sort_values("DateTime").reset_index(drop=True)

# 3) приберём NaN хотя бы в AddOn-колонках, чтобы они не ломали обучение
na_cols = ["AddOn_Anchor_Level", "AddOn_Anchor_IsUp", "AddOn_Size_Pct"]
exist_na_cols = [c for c in na_cols if c in df.columns]
if exist_na_cols:
    df = df.dropna(subset=exist_na_cols).reset_index(drop=True)

H = 20  # горизонт сделки

def add_goodtrade_target(df, horizon=20):
    df = df.copy()

    # будущая цена
    df["Close_fwd"] = df["Close"].shift(-horizon)

    # доходность по направлению сигнала
    ret_long  = (df["Close_fwd"] - df["Close"]) / df["Close"]
    ret_short = (df["Close"] - df["Close_fwd"]) / df["Close"]

    df["ret_H"] = np.where(
        df["EntrySignal"] > 0,  ret_long,
        np.where(df["EntrySignal"] < 0, ret_short, 0.0)
    )

    # GoodTrade: есть сигнал и ret_H > 0
    df["GoodTrade"] = ((df["EntrySignal"] != 0) & (df["ret_H"] > 0)).astype(int)

    # убираем хвост, где нет Close_fwd
    df = df.iloc[:-horizon].reset_index(drop=True)
    return df

df = add_goodtrade_target(df, horizon=H)

print(df[["DateTime", "Close", "Close_fwd", "EntrySignal", "ret_H", "GoodTrade"]].head())


              DateTime  Close  Close_fwd  EntrySignal    ret_H  GoodTrade
0  2015-10-26 19:00:00  47.86      47.08            0  0.00000          0
1  2015-10-26 20:00:00  47.65      47.10            0  0.00000          0
2  2015-10-26 21:00:00  47.57      47.16            0  0.00000          0
3  2015-10-26 22:00:00  47.47      47.17           -1  0.00632          1
4  2015-10-26 23:00:00  47.31      47.10            0  0.00000          0


In [ ]:
# только числовые колонки
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# что НЕ даём в признаки (таргет и будущее)
drop_feature_cols = ["Close_fwd", "ret_H", "GoodTrade"]
feature_cols = [c for c in numeric_cols if c not in drop_feature_cols]

print("Фичи для FEDformer-style:", feature_cols)

# сплит по времени 80/20
split_bar = int(len(df) * 0.8)
train_df = df.iloc[:split_bar].copy()
test_df  = df.iloc[split_bar:].copy()

print("Train bars:", len(train_df), "Test bars:", len(test_df))
print("Доля GoodTrade=1 train:", train_df["GoodTrade"].mean())
print("Доля GoodTrade=1 test :", test_df["GoodTrade"].mean())


Фичи для FEDformer-style: ['Open', 'High', 'Low', 'Close', 'Alligator_Jaw', 'Alligator_Teeth', 'Alligator_Lips', 'Fractal_Up', 'Fractal_Down', 'AO', 'Color AO', 'Alligator_Bullish', 'Alligator_Bearish', 'AlligatorStart_Long', 'AlligatorStart_Short', 'AO_sign', 'AO_zero_up', 'AO_zero_down', 'AO_three_green', 'AO_three_red', 'AO_saucer_up', 'AO_saucer_down', 'EntrySignal', 'Fractal_Up_conf', 'Fractal_Down_conf', 'AddOn_Anchor_Level', 'AddOn_Anchor_IsUp', 'AddOn_Size_Pct', 'AddOn_Ready', 'AddOn_Triggered']
Train bars: 28972 Test bars: 7243
Доля GoodTrade=1 train: 0.02423029131575314
Доля GoodTrade=1 test : 0.023747066132817893


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

SEQ_LEN = 96  # длина окна истории

def make_sequences(df_part, feature_cols, seq_len=96):
    data = df_part[feature_cols].values
    targets = df_part["GoodTrade"].values
    signals = df_part["EntrySignal"].values

    X_list, y_list = [], []

    for i in range(seq_len - 1, len(df_part)):
        # берём только те моменты, где на последнем баре окна есть сигнал
        if signals[i] == 0:
            continue

        X_seq = data[i - seq_len + 1 : i + 1, :]  # [seq_len, num_features]
        y_val = targets[i]

        X_list.append(X_seq)
        y_list.append(y_val)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y

X_train, y_train = make_sequences(train_df, feature_cols, seq_len=SEQ_LEN)
X_test, y_test   = make_sequences(test_df,  feature_cols, seq_len=SEQ_LEN)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Доля GoodTrade=1 в train seq:", (y_train == 1).mean())
print("Доля GoodTrade=1 в test seq :", (y_test == 1).mean())


X_train: (28877, 96, 30) X_test: (7148, 96, 30)
Доля GoodTrade=1 в train seq: 0.024240745229767637
Доля GoodTrade=1 в test seq : 0.02378287632904309


In [ ]:
class SeriesDecomp(tf.keras.layers.Layer):
    def __init__(self, kernel_size=25, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.avg_pool = AveragePooling1D(
            pool_size=kernel_size,
            strides=1,
            padding="same"
        )

    def call(self, x):
        # x: (batch, T, d)
        trend = self.avg_pool(x)
        seasonal = x - trend
        return seasonal, trend  # два тензора


In [ ]:
class FourierBlock(tf.keras.layers.Layer):
    def __init__(self, k=32, time_steps=96, **kwargs):
        super().__init__(**kwargs)
        self.k = k                 # сколько низких частот оставляем
        self.time_steps = time_steps  # длина окна по времени (SEQ_LEN)

    def call(self, x):
        # x: (batch, T, d)
        x = tf.cast(x, tf.float32)

        # переносим временную ось в конец: (batch, d, T)
        x_perm = tf.transpose(x, perm=[0, 2, 1])

        # FFT по последней оси (T)
        x_fft = tf.signal.rfft(x_perm)   # (batch, d, F)

        freq_len = tf.shape(x_fft)[-1]
        k = tf.minimum(self.k, freq_len)

        # оставляем только первые k частот, остальные обнуляем
        x_low = x_fft[..., :k]
        zeros = tf.zeros_like(x_fft[..., k:])
        x_trunc = tf.concat([x_low, zeros], axis=-1)  # (batch, d, F)

        # обратно во временную область; fft_length задаём позиционным аргументом
        x_rec = tf.signal.irfft(x_trunc, [self.time_steps])  # (batch, d, T)

        # возвращаем исходный порядок осей: (batch, T, d)
        out = tf.transpose(x_rec, perm=[0, 2, 1])
        return out

    def compute_output_shape(self, input_shape):
        return input_shape


In [ ]:
time_steps = SEQ_LEN
num_features = X_train.shape[2]
d_model = 64

inp = Input(shape=(time_steps, num_features))

# проекция фич
x = Dense(d_model, activation="linear")(inp)

# декомпозиция на seasonal + trend
seasonal, trend = SeriesDecomp(kernel_size=25)(x)  # оба: (batch, T, d_model)

# Fourier-обработка сезонности
seasonal_fft = FourierBlock(k=32)(seasonal)  # (batch, T, d_model)

# комбинируем обратно seasonal + trend
x_combined = Add()([seasonal_fft, trend])  # (batch, T, d_model)

# немного сгладим/усилим представление
x_combined = LayerNormalization(epsilon=1e-6)(x_combined)
x_combined = Dense(64, activation="relu")(x_combined)

# берём последний шаг по времени
x_last = Lambda(lambda t: t[:, -1, :])(x_combined)  # (batch, 64)

# классификатор
x_last = Dense(32, activation="relu")(x_last)
x_last = Dropout(0.3)(x_last)
out = Dense(1, activation="sigmoid")(x_last)

model = Model(inputs=inp, outputs=out)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 96, 30)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 96, 64)    │      1,984 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ series_decomp_4     │ [(None, 96, 64),  │          0 │ dense_16[0][0]    │
│ (SeriesDecomp)      │ (None, 96, 64)]   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fourier_block_4     │ (None, 96, 64)    │          0 │ series_decomp_4[… │
│ (FourierBlock)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 96, 64)    │          0 │ fourier_block_4[… │
│                     │                   │            │ series_decomp_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 96, 64)    │        128 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 96, 64)    │      4,160 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_4 (Lambda)   │ (None, 64)        │          0 │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 32)        │      2,080 │ lambda_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 32)        │          0 │ dense_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 1)         │         33 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,385 (32.75 KB)

 Trainable params: 8,385 (32.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
print("pos:", pos, "neg:", neg)

scale_pos = neg / pos if pos > 0 else 1.0
class_weight = {0: 1.0, 1: scale_pos}
print("class_weight:", class_weight)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weight,   # хочешь без учёта дисбаланса — убери этот аргумент
    verbose=1
)


pos: 700 neg: 28177
class_weight: {0: 1.0, 1: np.float64(40.252857142857145)}
Epoch 1/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 29s 55ms/step - accuracy: 0.8613 - loss: 0.6908 - val_accuracy: 0.9622 - val_loss: 0.1353
Epoch 2/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - accuracy: 0.9604 - loss: 0.2124 - val_accuracy: 0.9688 - val_loss: 0.1122
Epoch 3/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 19s 42ms/step - accuracy: 0.9641 - loss: 0.1899 - val_accuracy: 0.9659 - val_loss: 0.1285
Epoch 4/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 21s 47ms/step - accuracy: 0.9682 - loss: 0.1629 - val_accuracy: 0.9529 - val_loss: 0.1551
Epoch 5/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.9654 - loss: 0.1671 - val_accuracy: 0.9653 - val_loss: 0.1316
Epoch 6/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 21s 47ms/step - accuracy: 0.9689 - loss: 0.1538 - val_accuracy: 0.9668 - val_loss: 0.1300
Epoch 7/100
452/452 ━━━━━━━━━━━━━━━━━━━━ 21s 47ms/step - accuracy: 0.9704 - loss: 0.1405 - val_accuracy: 0.9717 - val_loss: 0.1117
Epoch

In [ ]:
y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("FEDformer-style AUC:", roc_auc_score(y_test, y_proba))
print("\nОтчёт по классификации (FEDformer-style):")
print(classification_report(y_test, y_pred, digits=3))
print("Матрица ошибок (FEDformer-style):")
print(confusion_matrix(y_test, y_pred))


224/224 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step
FEDformer-style AUC: 0.9869168647682633

Отчёт по классификации (FEDformer-style):
              precision    recall  f1-score   support

           0      0.999     0.974     0.986      6978
           1      0.474     0.976     0.638       170

    accuracy                          0.974      7148
   macro avg      0.737     0.975     0.812      7148
weighted avg      0.987     0.974     0.978      7148

Матрица ошибок (FEDformer-style):
[[6794  184]
 [   4  166]]
